# LLM Fine-tuning with LoRA
Fine-tune any HuggingFace causal language model using **LoRA** (Low-Rank Adaptation).

**Recommended runtime:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl sentencepiece scipy

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration
Change `MODEL_NAME` to any HuggingFace causal LM you want to fine-tune.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME     = "gpt2"          # e.g. "meta-llama/Llama-2-7b-hf", "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR     = "./lora-model"
LOAD_IN_4BIT   = False           # Set True for 7B+ models on free Colab GPU

# LoRA hyperparameters
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

# Training hyperparameters
EPOCHS         = 3
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
LEARNING_RATE  = 2e-4
MAX_SEQ_LEN    = 512

## Dataset
The dataset uses an instruction-input-output format. Add your own examples below.

In [ ]:
RAW_DATA = [
    {
        "instruction": "What is the capital of France?",
        "input": "",
        "output": "The capital of France is Paris."
    },
    {
        "instruction": "Explain what a neural network is.",
        "input": "",
        "output": "A neural network is a computational model inspired by the human brain. It consists of layers of interconnected nodes (neurons) that process and transform data. Each connection has a weight adjusted during training to minimize prediction errors."
    },
    {
        "instruction": "Summarize the following text.",
        "input": "The transformer architecture was introduced in 'Attention Is All You Need' by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms to draw global dependencies between input and output.",
        "output": "The transformer architecture, introduced in 2017, uses self-attention mechanisms instead of recurrence or convolutions to model dependencies in sequences."
    },
    {
        "instruction": "Write a Python function to reverse a string.",
        "input": "",
        "output": "def reverse_string(s: str) -> str:\n    return s[::-1]"
    },
    {
        "instruction": "What are the benefits of fine-tuning a language model?",
        "input": "",
        "output": "Fine-tuning adapts a general-purpose pretrained model to a specific task or domain with less data and compute. Benefits include improved domain performance and the ability to follow custom instructions."
    },
]

def format_prompt(ex):
    if ex.get("input"):
        return f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

dataset = Dataset.from_list([{"text": format_prompt(ex)} for ex in RAW_DATA])
print(f"Dataset: {len(dataset)} examples")
print("\nSample:")
print(dataset[0]["text"])

## Load Model & Tokenizer

In [ ]:
bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_NAME}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],  # for GPT-2 use ["c_attn"]
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

## Inference — Test Your Fine-tuned Model

In [ ]:
def generate_response(instruction, input_text="", max_new_tokens=200, temperature=0.7):
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Test the model
response = generate_response("What is the capital of France?")
print("Response:", response)

In [ ]:
# Interactive testing
while True:
    instruction = input("\nInstruction (or 'quit'): ")
    if instruction.lower() == "quit":
        break
    inp = input("Input (optional, press Enter to skip): ")
    response = generate_response(instruction, inp)
    print("\nResponse:", response)